# WANDS fine-tuning on Colab: LoRA r4, LoRA r16, full fine-tune (item 8)

All three training runs from the build plan, on one GPU, so their training times are comparable. The 8 GB MacBook killed the CPU training run at 96% (see PROGRESS.md 2026-10-29), so training moved here. Pairs, targets, seed, holdout, loss and hyperparameters are identical to `src/train_lora.py` in the repo.

After training, each model encodes all 42,994 products (same text as `src/encode.py`). Needs a GPU runtime. Produces `finetune_colab.zip` with, for each run: the merged model, the LoRA adapter (for its size), `metrics.json`, and the product vectors as `.npy`. Unzip into the repo so models land in `data/models/` and vectors in `data/vectors/`.

In [ ]:
%pip -q install sentence-transformers==3.4.1 datasets accelerate peft
!pip -q uninstall -y torchao  # Colab's torchao 0.10 is too old for peft; absent is fine
!git clone -q --depth 1 https://github.com/wayfair/WANDS.git wands_data
print("setup done")

In [ ]:
# Pair building and product text: identical to src/train_lora.py and src/encode.py.
import hashlib, random, time
import pandas as pd

PRODUCTS_CSV = "wands_data/dataset/product.csv"
BASE_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
SEED = 0
N_SAME_PATH = 10000
N_SAME_ROOT = 10000
N_CROSS = 10000
N_EVAL = 3000

def product_rows():
    p = pd.read_csv(PRODUCTS_CSV, sep="\t").fillna("")
    p = p.rename(columns={"category hierarchy": "category_hierarchy"})
    text = (p.product_name + ". " + p.product_class.str.replace("|", ", ", regex=False)
            + ". " + p.category_hierarchy.str.replace("/", ", ", regex=False)
            + ". " + p.product_description)
    path = [tuple(s.strip() for s in h.split("/") if s.strip())
            for h in p.category_hierarchy]
    return list(p.product_id), text.tolist(), path

def similarity(a, b):
    if not a or not b:
        return 0.0
    shared = 0
    for x, y in zip(a, b):
        if x != y:
            break
        shared += 1
    return shared / max(len(a), len(b))

def holdout(pid):
    return int(hashlib.md5(str(pid).encode()).hexdigest(), 16) % 10 == 0

def sample_pairs(idx, texts, paths, n_same_path, n_same_root, n_cross, rng):
    by_path, by_root = {}, {}
    for i in idx:
        if paths[i]:
            by_path.setdefault(paths[i], []).append(i)
            by_root.setdefault(paths[i][0], []).append(i)
    multi_path = [g for g in by_path.values() if len(g) > 1]
    multi_root = [g for g in by_root.values() if len(g) > 1]
    pairs = []
    for _ in range(n_same_path):
        pairs.append(tuple(rng.sample(rng.choice(multi_path), 2)))
    for _ in range(n_same_root):
        pairs.append(tuple(rng.sample(rng.choice(multi_root), 2)))
    for _ in range(n_cross):
        pairs.append(tuple(rng.sample(idx, 2)))
    return {"sentence1": [texts[a] for a, _ in pairs],
            "sentence2": [texts[b] for _, b in pairs],
            "score": [similarity(paths[a], paths[b]) for a, b in pairs]}

ids, texts, paths = product_rows()
train_idx = [i for i, pid in enumerate(ids) if not holdout(pid)]
eval_idx = [i for i, pid in enumerate(ids) if holdout(pid)]
print(len(train_idx), "train products,", len(eval_idx), "held out")

In [ ]:
# One function trains one run; all three use the same freshly sampled pairs (seed 0).
import json
from pathlib import Path

import numpy as np
import torch
from datasets import Dataset
from sentence_transformers import (SentenceTransformer, SentenceTransformerTrainer,
                                   SentenceTransformerTrainingArguments)
from sentence_transformers.losses import CosineSimilarityLoss

assert torch.cuda.is_available(), "Switch the runtime to a GPU first (Runtime > Change runtime type)."
DEVICE_NAME = torch.cuda.get_device_name(0)

rng = random.Random(SEED)
train_data = sample_pairs(train_idx, texts, paths, N_SAME_PATH, N_SAME_ROOT, N_CROSS, rng)
k = N_EVAL // 3
eval_data = sample_pairs(eval_idx, texts, paths, k, k, k, rng)
print(len(train_data["score"]), "train pairs,", len(eval_data["score"]), "eval pairs")

def dir_size_mb(path):
    return sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file()) / 1e6

def train_one(name, rank=None):
    torch.manual_seed(SEED)
    model = SentenceTransformer(BASE_MODEL, device="cuda")
    if rank is not None:
        from peft import LoraConfig, TaskType
        model.add_adapter(LoraConfig(task_type=TaskType.FEATURE_EXTRACTION,
                                     r=rank, lora_alpha=2 * rank, lora_dropout=0.1,
                                     target_modules=["query", "value"]))
    args = SentenceTransformerTrainingArguments(
        output_dir=f"{name}_trainer", num_train_epochs=1,
        per_device_train_batch_size=32, per_device_eval_batch_size=32,
        eval_strategy="no", save_strategy="no", logging_steps=100, seed=SEED, report_to=[],
    )
    trainer = SentenceTransformerTrainer(
        model=model, args=args,
        train_dataset=Dataset.from_dict(train_data),
        eval_dataset=Dataset.from_dict(eval_data),
        loss=CosineSimilarityLoss(model),
    )
    start = time.time()
    result = trainer.train()
    minutes = (time.time() - start) / 60
    eval_loss = trainer.evaluate()["eval_loss"]

    adapter_mb = None
    if rank is not None:
        model[0].auto_model.save_pretrained(f"out/{name}_adapter")
        adapter_mb = dir_size_mb(f"out/{name}_adapter")
        model[0].auto_model = model[0].auto_model.merge_and_unload()
    merged = f"out/{name}_merged"
    model.save(merged)

    vecs = model.encode(texts, batch_size=256, normalize_embeddings=True,
                        show_progress_bar=False)
    assert vecs.shape == (len(ids), 384) and not np.isnan(vecs).any()
    Path("out/vectors").mkdir(parents=True, exist_ok=True)
    np.save(f"out/vectors/products_{name}.npy", vecs.astype(np.float32))
    np.save(f"out/vectors/product_ids_{name}.npy", np.array(ids))

    metrics = {"mode": "lora" if rank else "full", "rank": rank, "base_model": BASE_MODEL,
               "epochs": 1, "batch_size": 32, "seed": SEED, "loss": "CosineSimilarityLoss",
               "train_pairs": len(train_data["score"]), "eval_pairs": len(eval_data["score"]),
               "train_loss": result.training_loss, "eval_loss": eval_loss,
               "train_minutes": minutes, "device": DEVICE_NAME,
               "saved_mb": dir_size_mb(merged), "adapter_mb": adapter_mb}
    Path(f"{merged}/metrics.json").write_text(json.dumps(metrics, indent=2))
    print(json.dumps(metrics, indent=2))

train_one("lora_r4", rank=4)

In [ ]:
train_one("lora_r16", rank=16)

In [ ]:
train_one("full_ft")

In [ ]:
# Zip everything and download. Unzip at the repo root: models go to data/models/,
# vectors to data/vectors/.
import subprocess, shutil
from pathlib import Path

stage = Path("finetune_colab")
if stage.exists():
    shutil.rmtree(stage)
(stage / "data/models").mkdir(parents=True)
(stage / "data/vectors").mkdir(parents=True)
for name in ["lora_r4", "lora_r16", "full_ft"]:
    shutil.copytree(f"out/{name}_merged", stage / f"data/models/{name}_merged")
    if Path(f"out/{name}_adapter").exists():
        shutil.copytree(f"out/{name}_adapter", stage / f"data/models/{name}_adapter")
    shutil.copy(f"out/vectors/products_{name}.npy", stage / "data/vectors/")
    shutil.copy(f"out/vectors/product_ids_{name}.npy", stage / "data/vectors/")
subprocess.run(["zip", "-qr", "finetune_colab.zip", "finetune_colab"], check=True)
print(f"{Path('finetune_colab.zip').stat().st_size / 1e6:.0f} MB")
from google.colab import files
files.download("finetune_colab.zip")